In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os


warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
    accuracy_score, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight

print("✅ All dependencies loaded successfully!")

✅ All dependencies loaded successfully!


In [23]:
df = pd.read_csv('../dataset/cleaned_data_with_risk.csv')
df.head()

,Age,Gender,University,Department,Academic_Year,CGPA,Scholarship,A1_Nervous,A2_Worrying,A3_Relaxing,...,D3_SleepTrouble,D4_Tired,D5_Appetite,D6_Failure,D7_Concentration,D8_Psychomotor,D9_SuicidalThoughts,Depression_Score,Depression_Label,Risk_Level
0,18-22,Female,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,Fourth Year or Equivalent,2.50 - 2.99,No,1,1,1,...,1,1,2,1,1,1,1,11,Moderate Depression,Moderate Risk
1,18-22,Male,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,First Year or Equivalent,3.80 - 4.00,No,2,2,1,...,1,1,1,1,1,1,1,9,Mild Depression,Moderate Risk
2,18-22,Male,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,First Year or Equivalent,3.00 - 3.39,No,2,1,1,...,2,3,2,2,2,2,1,16,Moderately Severe Depression,High Risk
3,18-22,Male,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,First Year or Equivalent,3.40 - 3.79,No,2,1,1,...,1,1,1,1,1,1,1,9,Mild Depression,Moderate Risk
4,18-22,Male,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,First Year or Equivalent,3.40 - 3.79,No,1,1,1,...,1,1,1,1,1,1,1,9,Mild Depression,Moderate Risk


In [24]:
regression_target = ['Anxiety_Score', 'Stress_Score', 'Depression_Score']
classification_target = ['Anxiety_Label', 'Stress_Label', 'Depression_Label','Risk_level']
feature_cols = [col for col in df.columns if col not in regression_target + classification_target]

In [25]:
print('Gender distribution before removal:')
gender_counts = df['Gender'].value_counts()
print(gender_counts)
df_cleaned = df[df['Gender'] != 'Prefer not to say'].copy()
df = df_cleaned
print(df['Gender'].value_counts())

Gender distribution before removal:
Gender
Male                 1372
Female                595
Prefer not to say      10
Name: count, dtype: int64
Gender
Male      1372
Female     595
Name: count, dtype: int64


In [26]:
ordinal_features = ['Age', 'Academic_Year', 'CGPA']
binary_features = ['Gender', 'Scholarship']

In [27]:
ordinal_encoder = OrdinalEncoder()
df[ordinal_features] = ordinal_encoder.fit_transform(df[ordinal_features])

In [28]:
# Binary mapping
binary_map = {
    'Gender': {'Male': 1, 'Female': 0},
    'Scholarship': {'Yes': 1, 'No': 0}
}

for feature, mapping in binary_map.items():
    df[feature] = df[feature].map(mapping)

In [29]:
X = df[feature_cols]
X_train , X_test = train_test_split(X, test_size=0.2, random_state=42)

## regression models

### Linear and logistic and random forest Regression for Anxiety, Stress and Depression score

In [30]:
y_anxiety = df['Anxiety_Score']
y_stress = df['Stress_Score']
y_depression = df['Depression_Score']

y_anxiety_train, y_anxiety_test = train_test_split(y_anxiety, test_size=0.2, random_state=42)
y_stress_train, y_stress_test = train_test_split(y_stress, test_size=0.2, random_state=42)
y_depression_train, y_depression_test = train_test_split(y_depression, test_size=0.2, random_state=42)

In [31]:
# anxiety score models
anxiety_model_linreg = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

anxiety_model_logreg = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LogisticRegression())
])

anxiety_model_rforest = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', RandomForestRegressor(n_estimators=200, random_state=42))
])

